# PrepSense — Data Retrieval Exploration
**Week 1 · Validating data sources before structuring the project**

Data sources used:
- **OpenWeatherMap One Call API 3.0** — current conditions + national alerts (AEMET for Spain)
- **OpenWeatherMap Forecast API 5 days/3h** — extended forecast (free tier)
- **SQLite** — local database schema validation

> One Call 3.0 requires the 'One Call by Call' subscription (free up to 1,000 calls/day).  
> Set your daily limit at: https://home.openweathermap.org/subscriptions

## 0. Setup

In [25]:
%pip install requests python-dotenv pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [26]:
import requests
import json
import sqlite3
import pandas as pd
from datetime import datetime
from pprint import pprint

print('Imports OK')

Imports OK


## 1. Configuration

Register your free API key at: https://openweathermap.org/api  
Activate One Call 3.0 at: https://home.openweathermap.org/subscriptions  

Load from a `.env` file

In [27]:
# Create a .env file with: OWM_API_KEY=your_key
from dotenv import load_dotenv
import os
load_dotenv()
OWM_API_KEY = os.getenv('OWM_API_KEY')

# Location
CITY  = 'Stade, Germany'
LAT   = 53.597573
LON   = 9.468527
UNITS = 'metric'

print(f'Config loaded | City: {CITY} | lat={LAT}, lon={LON}')

Config loaded | City: Stade, Germany | lat=53.597573, lon=9.468527


---
## 2. One Call API 3.0 — Current Conditions + National Alerts

Single endpoint returning current weather **and** active national alerts from AEMET (Spain).  
Docs: https://openweathermap.org/api/one-call-3

In [28]:
def get_onecall(api_key, lat, lon, units='metric'):
    """
    One Call API 3.0 — returns current conditions and national alerts.
    minutely/hourly/daily excluded to minimise payload.
    """
    url = 'https://api.openweathermap.org/data/3.0/onecall'
    params = {
        'lat':     lat,
        'lon':     lon,
        'appid':   api_key,
        'units':   units,
        'exclude': 'minutely,hourly,daily',
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


raw_onecall = get_onecall(OWM_API_KEY, LAT, LON)

print('One Call API 3.0 response received')
print(f'Keys in response: {list(raw_onecall.keys())}')
print(f'Active alerts:    {len(raw_onecall.get("alerts", []))}')

One Call API 3.0 response received
Keys in response: ['lat', 'lon', 'timezone', 'timezone_offset', 'current']
Active alerts:    0


In [29]:
def parse_current(raw):
    """Extract relevant fields from One Call current block."""
    c = raw.get('current', {})
    return {
        'timestamp':           datetime.utcfromtimestamp(c['dt']).isoformat(),
        'city':                raw.get('timezone', '').split('/')[-1],
        'temp_c':              c.get('temp'),
        'feels_like_c':        c.get('feels_like'),
        'humidity_pct':        c.get('humidity'),
        'pressure_hpa':        c.get('pressure'),
        'visibility_m':        c.get('visibility'),
        'wind_speed_ms':       c.get('wind_speed'),
        'wind_gust_ms':        c.get('wind_gust'),
        'rain_1h_mm':          c.get('rain', {}).get('1h', 0),
        'snow_1h_mm':          c.get('snow', {}).get('1h', 0),
        'weather_id':          c.get('weather', [{}])[0].get('id'),
        'weather_main':        c.get('weather', [{}])[0].get('main'),
        'weather_description': c.get('weather', [{}])[0].get('description'),
        'clouds_pct':          c.get('clouds'),
        'uvi':                 c.get('uvi'),
    }


weather = parse_current(raw_onecall)

print('Parsed current conditions:\n')
for k, v in weather.items():
    print(f'  {k:<28} {v}')

Parsed current conditions:

  timestamp                    2026-04-10T09:47:53
  city                         Berlin
  temp_c                       9.97
  feels_like_c                 7.66
  humidity_pct                 68
  pressure_hpa                 1016
  visibility_m                 10000
  wind_speed_ms                4.72
  wind_gust_ms                 6.29
  rain_1h_mm                   0
  snow_1h_mm                   0
  weather_id                   802
  weather_main                 Clouds
  weather_description          scattered clouds
  clouds_pct                   45
  uvi                          2.63


/tmp/ipykernel_38092/523335232.py:5: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'timestamp':           datetime.utcfromtimestamp(c['dt']).isoformat(),


In [30]:
# View as DataFrame
pd.DataFrame([weather]).T.rename(columns={0: 'value'})

,value
timestamp,2026-04-10T09:47:53
city,Berlin
temp_c,9.97
feels_like_c,7.66
humidity_pct,68
pressure_hpa,1016
visibility_m,10000
wind_speed_ms,4.72
wind_gust_ms,6.29
rain_1h_mm,0


---
## 3. One Call API 3.0 — National Alerts

Alerts are issued by **AEMET** for Spain.  

Note: One Call 3.0 does not expose severity as a label (Yellow/Orange/Red) directly —  
we infer it from the `tags` field and feed it into the risk engine.

In [31]:
def infer_severity_from_tags(tags):
    """
    Infer severity label from One Call alert tags.
    Returns: Minor / Moderate / Severe / Extreme / Unknown
    """
    tags_lower = [t.lower() for t in (tags or [])]
    if any(t in tags_lower for t in ['extreme', 'tornado', 'hurricane', 'tsunami']):
        return 'Extreme'
    elif any(t in tags_lower for t in ['thunderstorm', 'snow', 'flood', 'wind', 'rain', 'fog']):
        return 'Moderate'
    elif tags:
        return 'Minor'
    return 'Unknown'


def parse_alerts(raw):
    """Extract and normalise alerts from One Call 3.0 response."""
    parsed = []
    for a in raw.get('alerts', []):
        tags     = a.get('tags', [])
        severity = infer_severity_from_tags(tags)
        parsed.append({
            'title':       a.get('event', ''),
            'event':       a.get('event', ''),
            'sender_name': a.get('sender_name', ''),
            'severity':    severity,
            'urgency':     'Expected',
            'certainty':   'Likely',
            'onset':       datetime.utcfromtimestamp(a['start']).isoformat() if 'start' in a else '',
            'expires':     datetime.utcfromtimestamp(a['end']).isoformat()   if 'end'   in a else '',
            'description': a.get('description', ''),
            'tags':        tags,
        })
    return parsed


alerts = parse_alerts(raw_onecall)

print(f'Active alerts: {len(alerts)}')
if alerts:
    for a in alerts:
        print(f"\n  [{a['sender_name']}] {a['event']}")
        print(f"  Severity : {a['severity']}")
        print(f"  Tags     : {a['tags']}")
        print(f"  Onset    : {a['onset']}")
        print(f"  Expires  : {a['expires']}")
        print(f"  Desc     : {a['description'][:160]}")
else:
    print('  No active alerts — normal under calm conditions.')

Active alerts: 0
  No active alerts — normal under calm conditions.


In [32]:
# Show alerts as DataFrame
if alerts:
    display(pd.DataFrame(alerts).drop(columns=['description']))
else:
    cols = ['title', 'event', 'sender_name', 'severity', 'urgency', 'certainty', 'onset', 'expires', 'tags']
    print('No active alerts — empty DataFrame schema:')
    print(pd.DataFrame(columns=cols))

No active alerts — empty DataFrame schema:
Empty DataFrame
Columns: [title, event, sender_name, severity, urgency, certainty, onset, expires, tags]
Index: []


---
## 4. Forecast API — 5 Days / 3-Hour Steps

Separate free-tier endpoint — 40 timestamps over 5 days.  
Used for multi-day risk projection in the risk engine.  
Docs: https://openweathermap.org/forecast5

In [33]:
def get_forecast(api_key, lat, lon, units='metric'):
    """5-day / 3-hour forecast — free tier endpoint."""
    url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = {'lat': lat, 'lon': lon, 'appid': api_key, 'units': units}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


def parse_forecast(raw):
    """Convert forecast list to clean records."""
    records = []
    for item in raw['list']:
        weather_list = item.get('weather', [])

        # OWM returns a list but almost always with one element.
        # We take the first (most severe by convention) and log if there are more.
        if len(weather_list) > 1:
            print(f"[NOTE] Multiple weather conditions at {item['dt_txt']}: "
                  f"{[w['id'] for w in weather_list]} — using first.")

        primary_weather = weather_list[0] if weather_list else {}

        records.append({
            'timestamp':           item['dt_txt'],
            'temp_c':              item['main']['temp'],
            'temp_min_c':          item['main']['temp_min'],
            'temp_max_c':          item['main']['temp_max'],
            'humidity_pct':        item['main']['humidity'],
            'wind_speed_ms':       item['wind']['speed'],
            'wind_gust_ms':        item['wind'].get('gust'),
            'rain_3h_mm':          item.get('rain', {}).get('3h', 0),
            'snow_3h_mm':          item.get('snow', {}).get('3h', 0),
            'weather_id':          primary_weather.get('id'),
            'weather_main':        primary_weather.get('main'),
            'weather_description': primary_weather.get('description'),
            'weather_all_ids':     [w['id'] for w in weather_list],  # full list for reference
            'clouds_pct':          item['clouds']['all'],
            'pop':                 item.get('pop', 0),
        })
    return records


raw_forecast     = get_forecast(OWM_API_KEY, LAT, LON)
forecast_records = parse_forecast(raw_forecast)
df_forecast      = pd.DataFrame(forecast_records)
df_forecast['timestamp'] = pd.to_datetime(df_forecast['timestamp'])

print(f'Forecast received: {len(df_forecast)} timestamps')
print(f'Period: {df_forecast["timestamp"].iloc[0]} → {df_forecast["timestamp"].iloc[-1]}')
df_forecast.head(8)

Forecast received: 40 timestamps
Period: 2026-04-10 12:00:00 → 2026-04-15 09:00:00


,timestamp,temp_c,temp_min_c,temp_max_c,humidity_pct,wind_speed_ms,wind_gust_ms,rain_3h_mm,snow_3h_mm,weather_id,weather_main,weather_description,weather_all_ids,clouds_pct,pop
0,2026-04-10 12:00:00,10.48,10.48,11.50,65,5.35,6.00,0.0,0,803,Clouds,broken clouds,[803],54,0.0
1,2026-04-10 15:00:00,10.62,10.62,10.94,62,4.75,6.23,0.0,0,803,Clouds,broken clouds,[803],71,0.0
2,2026-04-10 18:00:00,8.47,8.47,8.47,84,2.78,4.88,0.0,0,802,Clouds,scattered clouds,[802],40,0.0
3,2026-04-10 21:00:00,5.34,5.34,5.34,93,0.72,0.72,0.0,0,800,Clear,clear sky,[800],4,0.0
4,2026-04-11 00:00:00,3.48,3.48,3.48,94,1.67,1.63,0.0,0,801,Clouds,few clouds,[801],11,0.0
5,2026-04-11 03:00:00,1.89,1.89,1.89,94,2.36,2.20,0.0,0,800,Clear,clear sky,[800],0,0.0
6,2026-04-11 06:00:00,3.58,3.58,3.58,88,2.86,7.05,0.0,0,800,Clear,clear sky,[800],2,0.0
7,2026-04-11 09:00:00,10.28,10.28,10.28,57,5.75,7.65,0.0,0,802,Clouds,scattered clouds,[802],33,0.0


In [35]:
# Apply severity score per timestamp first
df_forecast['severity_score'] = df_forecast['weather_id'].apply(weather_id_to_severity)

# Ensure date column exists (derived from timestamp)
df_forecast['date'] = df_forecast['timestamp'].dt.date

# Aggregate — worst day = highest severity score
daily_summary = df_forecast.groupby('date').agg(
    temp_min      =('temp_min_c',     'min'),
    temp_max      =('temp_max_c',     'max'),
    wind_max      =('wind_speed_ms',  'max'),
    rain_total    =('rain_3h_mm',     'sum'),
    snow_total    =('snow_3h_mm',     'sum'),
    max_pop       =('pop',            'max'),
    worst_severity=('severity_score', 'max'),
).reset_index()

# Recover the weather_id that produced the worst severity (for labelling)
worst_id_per_day = (
    df_forecast.sort_values('severity_score', ascending=False)
               .groupby('date', as_index=False)['weather_id']
               .first()
               .rename(columns={'weather_id': 'worst_weather_id'})
)

daily_summary = daily_summary.merge(worst_id_per_day, on='date')

print('Daily forecast summary:')
daily_summary

Daily forecast summary:


,date,temp_min,temp_max,wind_max,rain_total,snow_total,max_pop,worst_severity,worst_weather_id
0,2026-04-10,5.34,11.50,5.35,0.00,0,0.00,0,802
1,2026-04-11,1.89,13.27,7.16,0.15,0,0.35,15,500
2,2026-04-12,6.02,12.74,3.80,1.91,0,0.94,15,500
3,2026-04-13,4.77,8.48,4.66,7.80,0,1.00,20,501
4,2026-04-14,4.99,9.73,5.59,7.38,0,1.00,15,500
5,2026-04-15,3.11,9.85,3.23,0.00,0,0.00,0,802


---
## 5. Risk Score Engine — Preliminary

Combines OWM current conditions + One Call alerts into a 0-100 risk score.  
This logic will move to `core/risk_engine.py` in Week 2.

```
risk_score (0-100) = min(weather_severity + alert_severity + wind_bonus + rain_bonus, 100)
```

In [36]:
def severity_to_score(severity_str):
    """Map severity label to numeric score (0-60)."""
    return {'Minor': 10, 'Moderate': 25, 'Severe': 45, 'Extreme': 60}.get(severity_str, 0)


def score_to_level(score):
    """Convert numeric score to risk level label."""
    if score >= 70: return 'CRITICAL'
    if score >= 45: return 'HIGH'
    if score >= 20: return 'MEDIUM'
    return 'LOW'


def compute_risk_score(weather_data, alert_list):
    """
    Combine OWM current data + One Call alerts into a 0-100 risk score.
    Preliminary version — will be refined in Week 2.
    """
    weather_severity = weather_id_to_severity(weather_data['weather_id'])

    alert_severity = max(
        (severity_to_score(a['severity']) for a in alert_list),
        default=0
    )

    wind = weather_data.get('wind_speed_ms', 0) or 0
    wind_bonus = 15 if wind >= 20 else 8 if wind >= 14 else 3 if wind >= 8 else 0

    rain = weather_data.get('rain_1h_mm', 0) or 0
    rain_bonus = 10 if rain >= 20 else 5 if rain >= 10 else 0

    total = min(weather_severity + alert_severity + wind_bonus + rain_bonus, 100)

    return {
        'risk_score': total,
        'risk_level': score_to_level(total),
        'breakdown': {
            'weather_severity': weather_severity,
            'alert_severity':   alert_severity,
            'wind_bonus':       wind_bonus,
            'rain_bonus':       rain_bonus,
        }
    }


risk_result = compute_risk_score(weather, alerts)

print('=' * 42)
print(f"  RISK SCORE : {risk_result['risk_score']}/100")
print(f"  RISK LEVEL : {risk_result['risk_level']}")
print('=' * 42)
print('\nBreakdown:')
for k, v in risk_result['breakdown'].items():
    print(f'  {k:<22} +{v}')

  RISK SCORE : 0/100
  RISK LEVEL : LOW

Breakdown:
  weather_severity       +0
  alert_severity         +0
  wind_bonus             +0
  rain_bonus             +0


In [37]:
# Simulate scenarios to validate thresholds
print('Scenario simulation:\n')

scenarios = [
    {'name': 'Normal day',         'weather_id': 800, 'wind': 3,  'rain': 0,  'alert': 'Unknown'},
    {'name': 'Moderate rain',      'weather_id': 501, 'wind': 6,  'rain': 5,  'alert': 'Minor'},
    {'name': 'Storm + alert',      'weather_id': 502, 'wind': 18, 'rain': 15, 'alert': 'Moderate'},
    {'name': 'Severe emergency',   'weather_id': 212, 'wind': 25, 'rain': 25, 'alert': 'Severe'},
    {'name': 'Catastrophic event', 'weather_id': 781, 'wind': 35, 'rain': 40, 'alert': 'Extreme'},
]

for s in scenarios:
    fw = {'weather_id': s['weather_id'], 'wind_speed_ms': s['wind'], 'rain_1h_mm': s['rain']}
    fa = [{'severity': s['alert']}] if s['alert'] != 'Unknown' else []
    r  = compute_risk_score(fw, fa)
    print(f"  {s['name']:<26} score={r['risk_score']:>3}   level={r['risk_level']}")

Scenario simulation:

  Normal day                 score=  0   level=LOW
  Moderate rain              score= 30   level=MEDIUM
  Storm + alert              score= 68   level=HIGH
  Severe emergency           score=100   level=CRITICAL
  Catastrophic event         score=100   level=CRITICAL


---
## 6. Database Schema

In [38]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS weather_readings (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp           TEXT NOT NULL,
    city                TEXT,
    temp_c              REAL,
    feels_like_c        REAL,
    humidity_pct        INTEGER,
    pressure_hpa        INTEGER,
    visibility_m        INTEGER,
    wind_speed_ms       REAL,
    wind_gust_ms        REAL,
    rain_1h_mm          REAL DEFAULT 0,
    snow_1h_mm          REAL DEFAULT 0,
    weather_id          INTEGER,
    weather_main        TEXT,
    weather_desc        TEXT,
    clouds_pct          INTEGER,
    uvi                 REAL,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS weather_alerts (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    source              TEXT DEFAULT 'aemet',
    title               TEXT,
    event               TEXT,
    sender_name         TEXT,
    severity            TEXT,
    urgency             TEXT,
    certainty           TEXT,
    onset               TEXT,
    expires             TEXT,
    description         TEXT,
    tags                TEXT,
    is_active           INTEGER DEFAULT 1,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS risk_scores (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp           TEXT NOT NULL,
    score               INTEGER NOT NULL,
    level               TEXT NOT NULL,
    weather_component   INTEGER,
    alert_component     INTEGER,
    wind_bonus          INTEGER,
    rain_bonus          INTEGER,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS kit_items (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    name                TEXT NOT NULL,
    category            TEXT,
    quantity            REAL NOT NULL,
    unit                TEXT,
    expiry_date         TEXT,
    eu_recommended      REAL,
    notes               TEXT,
    updated_at          TEXT DEFAULT (datetime('now'))
);
"""

conn = sqlite3.connect('prepsense_test.db')
conn.executescript(SCHEMA_SQL)
conn.commit()

tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
print(f'Database created: prepsense_test.db')
print(f'Tables: {tables}')

Database created: prepsense_test.db
Tables: ['weather_readings', 'sqlite_sequence', 'weather_alerts', 'risk_scores', 'kit_items']


In [39]:
# Insert current weather reading
conn.execute("""
    INSERT INTO weather_readings
    (timestamp, city, temp_c, feels_like_c, humidity_pct, pressure_hpa,
     visibility_m, wind_speed_ms, wind_gust_ms, rain_1h_mm, snow_1h_mm,
     weather_id, weather_main, weather_desc, clouds_pct, uvi)
    VALUES
    (:timestamp, :city, :temp_c, :feels_like_c, :humidity_pct, :pressure_hpa,
     :visibility_m, :wind_speed_ms, :wind_gust_ms, :rain_1h_mm, :snow_1h_mm,
     :weather_id, :weather_main, :weather_description, :clouds_pct, :uvi)
""", weather)

# Insert alerts
for a in alerts:
    conn.execute("""
        INSERT INTO weather_alerts
        (title, event, sender_name, severity, urgency, certainty, onset, expires, description, tags)
        VALUES (:title, :event, :sender_name, :severity, :urgency, :certainty,
                :onset, :expires, :description, :tags)
    """, {**a, 'tags': json.dumps(a['tags'])})

# Insert risk score
conn.execute("""
    INSERT INTO risk_scores
    (timestamp, score, level, weather_component, alert_component, wind_bonus, rain_bonus)
    VALUES (datetime('now'), :risk_score, :risk_level,
            :weather_severity, :alert_severity, :wind_bonus, :rain_bonus)
""", {
    'risk_score':       risk_result['risk_score'],
    'risk_level':       risk_result['risk_level'],
    **risk_result['breakdown']
})

conn.commit()
print('Data inserted into DB')

row = conn.execute('SELECT timestamp, temp_c, wind_speed_ms, weather_desc FROM weather_readings LIMIT 1').fetchone()
print(f'Weather row: {row}')

score_row = conn.execute('SELECT score, level FROM risk_scores LIMIT 1').fetchone()
print(f'Risk score: {score_row[0]}/100 ({score_row[1]})')

Data inserted into DB
Weather row: ('2026-04-10T09:47:53', 9.97, 4.72, 'scattered clouds')
Risk score: 0/100 (LOW)


---
## 7. Emergency Kit — Seed Data

Reference kit based on EU emergency preparedness recommendations (per person, 3 days).  
Source: https://ec.europa.eu/echo/what/civil-protection/eu-emergency-preparedness

In [40]:
EU_KIT_REFERENCE = [
    {'name': 'Drinking water',          'category': 'water',     'eu_recommended': 9.0, 'unit': 'liters'},
    {'name': 'Water purification tabs', 'category': 'water',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Non-perishable food',     'category': 'food',      'eu_recommended': 3.0, 'unit': 'days'},
    {'name': 'Can opener',              'category': 'food',      'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'First aid kit',           'category': 'meds',      'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Regular medication',      'category': 'meds',      'eu_recommended': 7.0, 'unit': 'days'},
    {'name': 'Flashlight',              'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'AA batteries',            'category': 'tools',     'eu_recommended': 6.0, 'unit': 'units'},
    {'name': 'Battery-powered radio',   'category': 'comms',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Power bank',              'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Whistle',                 'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Sleeping bag',            'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Emergency blanket',       'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Document copies',         'category': 'documents', 'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Cash',                    'category': 'documents', 'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Basic hygiene kit',       'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
]

# Simulate a realistic kit with intentional gaps and expiry dates
current_kit = [{**item, 'quantity': item['eu_recommended'] * 0.6, 'expiry_date': None}
               for item in EU_KIT_REFERENCE]

current_kit[0]['quantity']    = 4.0           # Water: 4L vs 9L — gap!
current_kit[2]['expiry_date'] = '2026-04-20'  # Food expiring soon
current_kit[5]['expiry_date'] = '2027-01-01'  # Medication OK

for item in current_kit:
    conn.execute("""
        INSERT INTO kit_items (name, category, quantity, unit, expiry_date, eu_recommended)
        VALUES (:name, :category, :quantity, :unit, :expiry_date, :eu_recommended)
    """, item)
conn.commit()

print(f'{len(current_kit)} kit items inserted\n')

df_kit = pd.read_sql(
    'SELECT name, category, quantity, eu_recommended, unit, expiry_date FROM kit_items', conn
)
df_kit['gap']     = df_kit['eu_recommended'] - df_kit['quantity']
df_kit['gap_pct'] = (df_kit['gap'] / df_kit['eu_recommended'] * 100).round(1)
df_kit

16 kit items inserted



,name,category,quantity,eu_recommended,unit,expiry_date,gap,gap_pct
0,Drinking water,water,4.0,9.0,liters,NaN,5.0,55.6
1,Water purification tabs,water,0.6,1.0,units,NaN,0.4,40.0
2,Non-perishable food,food,1.8,3.0,days,2026-04-20,1.2,40.0
3,Can opener,food,0.6,1.0,units,NaN,0.4,40.0
4,First aid kit,meds,0.6,1.0,units,NaN,0.4,40.0
5,Regular medication,meds,4.2,7.0,days,2027-01-01,2.8,40.0
6,Flashlight,tools,0.6,1.0,units,NaN,0.4,40.0
7,AA batteries,tools,3.6,6.0,units,NaN,2.4,40.0
8,Battery-powered radio,comms,0.6,1.0,units,NaN,0.4,40.0
9,Power bank,tools,0.6,1.0,units,NaN,0.4,40.0


In [41]:
today = datetime.today()

print('KIT GAPS (below EU recommendation):')
for _, row in df_kit[df_kit['gap'] > 0].iterrows():
    print(f"  {row['name']:<32} {row['quantity']:.1f} / {row['eu_recommended']:.1f} {row['unit']} ({row['gap_pct']}% missing)")

print('\nEXPIRING ITEMS (within 30 days):')
expiring = df_kit[df_kit['expiry_date'].notna()].copy()
expiring['days_to_expiry'] = expiring['expiry_date'].apply(
    lambda d: (datetime.strptime(d, '%Y-%m-%d') - today).days
)
critical = expiring[expiring['days_to_expiry'] <= 30]
for _, row in critical.iterrows():
    flag = 'CRITICAL' if row['days_to_expiry'] <= 7 else 'WARNING'
    print(f"  [{flag}] {row['name']:<32} expires in {row['days_to_expiry']} days ({row['expiry_date']})")
if critical.empty:
    print('  No items expiring within 30 days.')

KIT GAPS (below EU recommendation):
  Drinking water                   4.0 / 9.0 liters (55.6% missing)
  Water purification tabs          0.6 / 1.0 units (40.0% missing)
  Non-perishable food              1.8 / 3.0 days (40.0% missing)
  Can opener                       0.6 / 1.0 units (40.0% missing)
  First aid kit                    0.6 / 1.0 units (40.0% missing)
  Regular medication               4.2 / 7.0 days (40.0% missing)
  Flashlight                       0.6 / 1.0 units (40.0% missing)
  AA batteries                     3.6 / 6.0 units (40.0% missing)
  Battery-powered radio            0.6 / 1.0 units (40.0% missing)
  Power bank                       0.6 / 1.0 units (40.0% missing)
  Whistle                          0.6 / 1.0 units (40.0% missing)
  Sleeping bag                     0.6 / 1.0 units (40.0% missing)
  Emergency blanket                0.6 / 1.0 units (40.0% missing)
  Document copies                  0.6 / 1.0 units (40.0% missing)
  Cash                     

---
## 8. Summary

In [42]:
print('=' * 55)
print('  PREPSENSE — DATA RETRIEVAL VALIDATION SUMMARY')
print('=' * 55)

print('\n[OK] One Call API 3.0 — current conditions')
print(f"     Temp: {weather['temp_c']}C | Wind: {weather['wind_speed_ms']}m/s | {weather['weather_description']}")

print('\n[OK] One Call API 3.0 — national alerts (AEMET)')
print(f'     {len(alerts)} active alert(s)')

print('\n[OK] Forecast API — 5 days / 3h')
print(f'     {len(df_forecast)} timestamps | {len(daily_summary)} days aggregated')

print('\n[OK] Risk Score Engine (preliminary)')
print(f"     Score: {risk_result['risk_score']}/100 ({risk_result['risk_level']})")

print('\n[OK] SQLite DB — schema validated')
print(f'     Tables: weather_readings, weather_alerts, risk_scores, kit_items')

print(f'\n[OK] Emergency kit — {len(current_kit)} items')
gaps_count = len(df_kit[df_kit['gap'] > 0])
print(f'     {gaps_count} gap(s) | {len(critical)} item(s) expiring within 30 days')

print('\n' + '=' * 55)
print('  NEXT STEPS (Week 2)')
print('=' * 55)
print("""
  1. Move fetchers to data/fetchers/ as Python modules
  2. Create data/db.py with DB access functions
  3. Build core/risk_engine.py with logic validated here
  4. Build core/inventory_analyzer.py (gap + expiry detection)
  5. Build core/alert_prioritizer.py (risk x gaps -> ranked actions)
""")

conn.close()
print('DB connection closed.')

  PREPSENSE — DATA RETRIEVAL VALIDATION SUMMARY

[OK] One Call API 3.0 — current conditions
     Temp: 9.97C | Wind: 4.72m/s | scattered clouds

[OK] One Call API 3.0 — national alerts (AEMET)
     0 active alert(s)

[OK] Forecast API — 5 days / 3h
     40 timestamps | 6 days aggregated

[OK] Risk Score Engine (preliminary)
     Score: 0/100 (LOW)

[OK] SQLite DB — schema validated
     Tables: weather_readings, weather_alerts, risk_scores, kit_items

[OK] Emergency kit — 16 items
     16 gap(s) | 1 item(s) expiring within 30 days

  NEXT STEPS (Week 2)

  1. Move fetchers to data/fetchers/ as Python modules
  2. Create data/db.py with DB access functions
  3. Build core/risk_engine.py with logic validated here
  4. Build core/inventory_analyzer.py (gap + expiry detection)
  5. Build core/alert_prioritizer.py (risk x gaps -> ranked actions)

DB connection closed.
